# Morphometrics Analysis (ADS 2015 — No QC)

This notebook uses the **original** pre-existing morphometrics files found in the `_corrections/` folders. These were computed with an earlier version of AxonDeepSeg without manual quality control corrections and do not include border-touching information.

## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from pathlib import Path
from IPython.display import display
import plotly.graph_objects as go

In [2]:
DATA_DIR = Path("data/macaque/data")
PLOT_TEMPLATE = "plotly_white"

SAMPLE_SLICES = {
    "sample1": ["01", "03", "07"],
    "sample2": ["01", "03", "05"],
    "sample3": ["01", "03", "05"],
    "sample4": ["01", "05", "07"],
    "sample5": ["01", "03", "05"],
    "sample6": ["01", "03", "05"],
    "sample7": ["01", "03", "05"],
    "sample8": ["01", "03", "05"],
}

# sample4/07 has a different filename
OLD_XLSX_NAMES = {
    ("sample4", "07"): "07 titre.xlsx",
}

REGION_LABELS = {
    "sample1": "Region 1",
    "sample2": "Region 2",
    "sample3": "Region 3",
    "sample4": "Region 4",
    "sample5": "Region 5",
    "sample6": "Region 6",
    "sample7": "Region 7",
    "sample8": "Region 8",
}

## Load and filter morphometrics

These files use the old column naming (no units in headers, no `image_border_touching` column). Filtering is limited to removing unmyelinated axons and invalid g-ratios.

In [3]:
def load_old_morphometrics(sample, slice_id):
    """Load an old morphometrics xlsx from the corrections folder."""
    fname = OLD_XLSX_NAMES.get((sample, slice_id), f"{slice_id}.xlsx")
    path = DATA_DIR / sample / f"{slice_id}_corrections" / fname
    df = pd.read_excel(path)
    # Drop the unnamed index column if present
    df = df.drop(columns=[c for c in df.columns if "Unnamed" in str(c)], errors="ignore")
    df["sample"] = sample
    df["slice_id"] = slice_id
    return df


def filter_old_morphometrics(df):
    """Filter without border-touching info."""
    df = df[df["myelin_area"] > 0]
    df = df[df["myelin_thickness"] > 0]
    df = df[df["gratio"] < 1.0]
    df = df[df["axon_area"] > 0]
    df = df[df["axon_diam"] > 0]
    df = df[df["gratio"] > 0]
    df = df.dropna(subset=["gratio", "axon_diam", "myelin_thickness", "myelin_area"])
    return df

In [4]:
# Load, filter, and combine morphometrics for each sample
# Aggregate g-ratio per image: g_agg = sqrt(sum(axon_area) / (sum(axon_area) + sum(myelin_area)))
all_data = {}
agg_gratio = {}
filter_summary = []

for sample in sorted(SAMPLE_SLICES.keys()):
    dfs = []
    sample_agg = []
    for slice_id in SAMPLE_SLICES[sample]:
        df = load_old_morphometrics(sample, slice_id)
        n_before = len(df)

        # Compute aggregate g-ratio from ALL axons (before filtering)
        total_axon = df["axon_area"].sum()
        total_myelin = df["myelin_area"].sum()
        if total_axon + total_myelin > 0:
            g_agg = np.sqrt(total_axon / (total_axon + total_myelin))
        else:
            g_agg = np.nan
        sample_agg.append(g_agg)

        df = filter_old_morphometrics(df)
        n_after = len(df)
        filter_summary.append({
            "Region": REGION_LABELS[sample],
            "Slice": slice_id,
            "Before": n_before,
            "After": n_after,
            "Removed": n_before - n_after,
        })
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)
    combined["fiber_diam"] = combined["axon_diam"] + 2 * combined["myelin_thickness"]
    all_data[sample] = combined
    agg_gratio[sample] = sample_agg

display(pd.DataFrame(filter_summary))

,Region,Slice,Before,After,Removed
0,Region 1,01,496,495,1
1,Region 1,03,445,442,3
2,Region 1,07,496,495,1
3,Region 2,01,308,307,1
4,Region 2,03,369,367,2
5,Region 2,05,364,363,1
6,Region 3,01,276,271,5
7,Region 3,03,166,156,10
8,Region 3,05,276,274,2
9,Region 4,01,144,131,13


## Summary statistics

The **aggregate g-ratio** is computed per image by summing axon and myelin areas from the morphometrics: $g_{\text{agg}} = \sqrt{\Sigma A_{\text{axon}} / (\Sigma A_{\text{axon}} + \Sigma A_{\text{myelin}})}$ (Stikov et al. 2015), then averaged across images within each region.

In [5]:
summary_rows = []
for sample in sorted(all_data.keys()):
    df = all_data[sample]
    ag = agg_gratio[sample]
    summary_rows.append({
        "Region": REGION_LABELS[sample],
        "n": len(df),
        "Axon diam (\u03bcm)": f"{df['axon_diam'].mean():.4f} \u00b1 {df['axon_diam'].std():.4f}",
        "Fiber diam (\u03bcm)": f"{df['fiber_diam'].mean():.4f} \u00b1 {df['fiber_diam'].std():.4f}",
        "G-ratio (per-axon)": f"{df['gratio'].mean():.4f} \u00b1 {df['gratio'].std():.4f}",
        "G-ratio (aggregate)": f"{np.mean(ag):.4f} \u00b1 {np.std(ag):.4f}",
        "Myelin thickness (\u03bcm)": f"{df['myelin_thickness'].mean():.4f} \u00b1 {df['myelin_thickness'].std():.4f}",
    })

display(pd.DataFrame(summary_rows))

,Region,n,Axon diam (μm),Fiber diam (μm),G-ratio (per-axon),G-ratio (aggregate),Myelin thickness (μm)
0,Region 1,1432,0.7047 ± 0.3334,0.9220 ± 0.3640,0.7390 ± 0.1095,0.7859 ± 0.0213,0.1086 ± 0.0360
1,Region 2,1037,0.7477 ± 0.4572,0.9702 ± 0.5105,0.7354 ± 0.1306,0.7992 ± 0.0128,0.1113 ± 0.0506
2,Region 3,701,0.7921 ± 0.6281,1.0301 ± 0.7274,0.7110 ± 0.1775,0.8013 ± 0.0068,0.1190 ± 0.0725
3,Region 4,472,0.8121 ± 0.6905,1.0541 ± 0.7996,0.6638 ± 0.2486,0.8166 ± 0.0168,0.1210 ± 0.1156
4,Region 5,705,0.8859 ± 0.8316,1.1766 ± 0.9258,0.6816 ± 0.1980,0.8107 ± 0.0154,0.1453 ± 0.0817
5,Region 6,874,0.7545 ± 0.6121,1.0829 ± 0.7110,0.6321 ± 0.1868,0.7513 ± 0.0249,0.1642 ± 0.0790
6,Region 7,961,0.7034 ± 0.5475,0.9848 ± 0.6182,0.6565 ± 0.1723,0.7671 ± 0.0121,0.1407 ± 0.0653
7,Region 8,532,0.8459 ± 0.7620,1.0944 ± 0.8502,0.6833 ± 0.2131,0.8310 ± 0.0209,0.1243 ± 0.0842


## Axon diameter vs Fiber diameter

Use the slider to switch between CC regions.

In [6]:
samples_sorted = sorted(all_data.keys())
all_df = pd.concat(all_data.values())
x_max = all_df["axon_diam"].max() * 1.05
y_max = all_df["fiber_diam"].max() * 1.05

fig = go.Figure()

for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["axon_diam"].values, df["fiber_diam"].values
    visible = (i == 0)

    fig.add_trace(go.Scattergl(
        x=x, y=y, mode="markers",
        marker=dict(size=4, color="steelblue", opacity=0.3),
        showlegend=False, visible=visible,
        hovertemplate="Axon: %{x:.3f} \u03bcm<br>Fiber: %{y:.3f} \u03bcm<extra></extra>",
    ))

    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)
    x_line = np.linspace(0, x_max, 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=slope * x_line + intercept, mode="lines",
        line=dict(color="red", width=2), showlegend=False, visible=visible,
        hovertemplate=f"y = {slope:.2f}x + {intercept:.4f}<br>R\u00b2 = {r_value**2:.3f}<extra></extra>",
    ))

n_traces = len(samples_sorted) * 2
steps = []
for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["axon_diam"].values, df["fiber_diam"].values
    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)

    vis = [False] * n_traces
    vis[i * 2] = True
    vis[i * 2 + 1] = True
    step = dict(
        method="update",
        args=[
            {"visible": vis},
            {"title": f"{REGION_LABELS[sample]} (n = {len(df)})",
             "annotations": [
                 dict(text=f"y = {slope:.2f}x + {intercept:.4f}<br>R\u00b2 = {r_value**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.95,
                      xanchor="left", yanchor="top", showarrow=False,
                      font=dict(size=14, color="red")),
             ]},
        ],
        label=REGION_LABELS[sample],
    )
    steps.append(step)

df0 = all_data[samples_sorted[0]]
s0, i0, r0, _, _ = scipy_stats.linregress(df0["axon_diam"].values, df0["fiber_diam"].values)

fig.update_layout(
    sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="Region: "), pad=dict(t=50))],
    title=f"{REGION_LABELS[samples_sorted[0]]} (n = {len(df0)})",
    xaxis_title="Axon diameter (\u03bcm)",
    yaxis_title="Fiber diameter (\u03bcm)",
    xaxis_range=[0, x_max], yaxis_range=[0, y_max],
    template=PLOT_TEMPLATE, height=600, width=800,
    annotations=[dict(text=f"y = {s0:.2f}x + {i0:.4f}<br>R\u00b2 = {r0**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.95,
                      xanchor="left", yanchor="top", showarrow=False,
                      font=dict(size=14, color="red"))],
)
fig.show()

## G-ratio vs Fiber diameter

Use the slider to switch between CC regions.

In [7]:
x_max_g = all_df["fiber_diam"].max() * 1.05
y_min_g = max(all_df["gratio"].min() * 0.95, 0)
y_max_g = min(all_df["gratio"].max() * 1.05, 1.0)

fig = go.Figure()

for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["fiber_diam"].values, df["gratio"].values
    visible = (i == 0)

    fig.add_trace(go.Scattergl(
        x=x, y=y, mode="markers",
        marker=dict(size=4, color="darkorange", opacity=0.3),
        showlegend=False, visible=visible,
        hovertemplate="Fiber: %{x:.3f} \u03bcm<br>G-ratio: %{y:.3f}<extra></extra>",
    ))

    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)
    x_line = np.linspace(0, x_max_g, 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=slope * x_line + intercept, mode="lines",
        line=dict(color="red", width=2), showlegend=False, visible=visible,
        hovertemplate=f"y = {slope:.4f}x + {intercept:.3f}<br>R\u00b2 = {r_value**2:.3f}<extra></extra>",
    ))

n_traces = len(samples_sorted) * 2
steps = []
for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["fiber_diam"].values, df["gratio"].values
    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)

    vis = [False] * n_traces
    vis[i * 2] = True
    vis[i * 2 + 1] = True
    step = dict(
        method="update",
        args=[
            {"visible": vis},
            {"title": f"{REGION_LABELS[sample]} (n = {len(df)})",
             "annotations": [
                 dict(text=f"y = {slope:.4f}x + {intercept:.3f}<br>R\u00b2 = {r_value**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.05,
                      xanchor="left", yanchor="bottom", showarrow=False,
                      font=dict(size=14, color="red")),
             ]},
        ],
        label=REGION_LABELS[sample],
    )
    steps.append(step)

df0 = all_data[samples_sorted[0]]
s0, i0, r0, _, _ = scipy_stats.linregress(df0["fiber_diam"].values, df0["gratio"].values)

fig.update_layout(
    sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="Region: "), pad=dict(t=50))],
    title=f"{REGION_LABELS[samples_sorted[0]]} (n = {len(df0)})",
    xaxis_title="Fiber diameter (\u03bcm)",
    yaxis_title="G-ratio",
    xaxis_range=[0, x_max_g], yaxis_range=[y_min_g, y_max_g],
    template=PLOT_TEMPLATE, height=600, width=800,
    annotations=[dict(text=f"y = {s0:.4f}x + {i0:.3f}<br>R\u00b2 = {r0**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.05,
                      xanchor="left", yanchor="bottom", showarrow=False,
                      font=dict(size=14, color="red"))],
)
fig.show()